# M4 — Constraint Programming com CP-SAT

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Caso: Escala de Plantão do Suporte L1 da Gradus

O time de Suporte L1 cobre **segunda a sexta, 4 turnos/dia**:

| Turno | Horário |
|---|---|
| T1 | 08–12h |
| T2 | 12–16h |
| T3 | 16–20h |
| T4 | 20–24h |

São **8 analistas**. Fechar a escala respeitando:

1. Cada turno é coberto por exatamente 1 analista
2. Ninguém faz 2 turnos no mesmo dia
3. Sem T4 (noite) → T1 (manhã) no dia seguinte (descanso)
4. Máximo 4 turnos por semana por analista
5. Indisponibilidades individuais (curso, compromisso)
6. Senior só faz T3 ou T4 (cobertura fora do horário comercial)

Em MIP, regras 2, 3 e 6 viram big-M soup. Em CP-SAT, viram 3 linhas naturais.

## Setup — CP-SAT vs Gurobi MIP

**Constraint Programming (CP)** e **Mixed-Integer Programming (MIP)** são duas tecnologias diferentes:

- **CP-SAT** (Google OR-Tools) propaga restrições simbolicamente. Brilha em problemas combinatórios discretos (scheduling, rostering, packing).
- **Gurobi MIP** resolve via branch-and-cut com relaxações lineares. Brilha em otimização contínua + binárias com estrutura LP-amigável.

Neste módulo modelamos a escala de plantão **nos dois**. Spoiler: CP-SAT vence em tempo e legibilidade neste tipo de problema. Mas a versão Gurobi é útil quando a única licença disponível é a do Gurobi (cenário típico em consultoria com cliente que já investiu).

In [ ]:
%pip install -q ortools gurobipy

In [ ]:
from ortools.sat.python import cp_model
from itertools import product

ANALISTAS = ['Ana', 'Bruno', 'Carla', 'Diego', 'Eduarda', 'Fábio', 'Gabi', 'Hugo']
SENIOR    = {'Eduarda', 'Hugo'}
DIAS      = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex']
TURNOS    = ['T1', 'T2', 'T3', 'T4']

INDISP = [
    ('Ana',   'Qua', 'T1'),   # curso de inglês
    ('Bruno', 'Ter', 'T2'),   # PJ no escritório
    ('Carla', 'Qui', 'T3'),   # consulta médica
    ('Diego', 'Sex', 'T4'),   # compromisso pessoal
]

N, D, T = len(ANALISTAS), len(DIAS), len(TURNOS)
print(f'{N} analistas × {D} dias × {T} turnos = {N*D*T} variáveis booleanas')

## Modelo CP-SAT — caso simples

In [ ]:
def montar_modelo_simples():
    m = cp_model.CpModel()
    x = {(a,d,t): m.NewBoolVar(f'x[{a},{d},{t}]')
         for a in range(N) for d in range(D) for t in range(T)}

    # (1) Cobertura: exatamente 1 analista por turno
    for d, t in product(range(D), range(T)):
        m.AddExactlyOne([x[a, d, t] for a in range(N)])

    # (2) Máximo 1 turno por dia por analista
    for a, d in product(range(N), range(D)):
        m.AddAtMostOne([x[a, d, t] for t in range(T)])

    # (3) Sem T4 → T1 no dia seguinte
    for a, d in product(range(N), range(D - 1)):
        m.Add(x[a, d, 3] + x[a, d+1, 0] <= 1)

    # (4) Máximo 4 turnos por semana
    for a in range(N):
        m.Add(sum(x[a, d, t] for d in range(D) for t in range(T)) <= 4)

    # (5) Indisponibilidades
    for ana, dia, tur in INDISP:
        a = ANALISTAS.index(ana); d = DIAS.index(dia); t = TURNOS.index(tur)
        m.Add(x[a, d, t] == 0)

    # (6) Senior só faz T3 ou T4
    for ana in SENIOR:
        a = ANALISTAS.index(ana)
        for d in range(D):
            m.Add(x[a, d, 0] == 0)   # nem T1
            m.Add(x[a, d, 1] == 0)   # nem T2

    return m, x

def imprimir_escala(solver, x, dias, turnos, analistas):
    print(f"{'Analista':<10} | " + " | ".join(f"{d:^6}" for d in dias))
    print('-' * (12 + 9 * len(dias)))
    for a, ana in enumerate(analistas):
        row = [f"{ana:<10}"]
        for d in range(len(dias)):
            cell = ''
            for t in range(len(turnos)):
                if (a, d, t) in x.indices() if hasattr(x, 'indices') else True:
                    if (a, d, t) in x and solver.Value(x[a, d, t]) == 1:
                        cell = turnos[t]
            row.append(f"{cell:^6}")
        print(' | '.join(row))
    cargas = [sum(solver.Value(x[a,d,t]) for d in range(len(dias)) for t in range(len(turnos)) if (a,d,t) in x) for a in range(len(analistas))]
    print(f"\nCargas: {dict(zip(analistas, cargas))}")

m, x = montar_modelo_simples()
solver = cp_model.CpSolver()
solver.parameters.num_search_workers = 4
status = solver.Solve(m)
print(f'Status: {solver.StatusName(status)}  ({solver.WallTime()*1000:.1f} ms)')
imprimir_escala(solver, x, DIAS, TURNOS, ANALISTAS)

---

## O mesmo problema em Gurobi (MIP)

Modelagem MIP do mesmo problema. Mesmas variáveis ($x_{a,d,t}$ binárias), mas as restrições "AddExactlyOne" e "AddAtMostOne" do CP-SAT viram desigualdades algébricas:

| CP-SAT | MIP equivalente |
|---|---|
| `AddExactlyOne([v1,v2,v3])` | `v1 + v2 + v3 == 1` |
| `AddAtMostOne([v1,v2,v3])` | `v1 + v2 + v3 <= 1` |
| `Add(a + b <= 1)` | igual |
| `m.Add(x).OnlyEnforceIf(y)` | precisa big-M |

Para *esse* problema, todas as restrições são lineares simples — nenhum big-M necessário. Então MIP roda bem. O ponto frágil do MIP aparece quando há restrições condicionais ("se X então Y") — aí o big-M é difícil de calibrar.

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import time

def montar_modelo_gurobi():
    m = gp.Model('escala_simples')
    m.Params.OutputFlag = 0

    x = m.addVars(N, D, T, vtype=GRB.BINARY, name='x')

    # (1) Cobertura: exatamente 1 analista por turno
    m.addConstrs((gp.quicksum(x[a, d, t] for a in range(N)) == 1
                  for d in range(D) for t in range(T)), name='cob')

    # (2) Máximo 1 turno por dia por analista
    m.addConstrs((gp.quicksum(x[a, d, t] for t in range(T)) <= 1
                  for a in range(N) for d in range(D)), name='1pordia')

    # (3) Sem T4 → T1 no dia seguinte
    m.addConstrs((x[a, d, 3] + x[a, d+1, 0] <= 1
                  for a in range(N) for d in range(D-1)), name='descanso')

    # (4) Máximo 4 turnos por semana
    m.addConstrs((gp.quicksum(x[a, d, t] for d in range(D) for t in range(T)) <= 4
                  for a in range(N)), name='maxsem')

    # (5) Indisponibilidades
    for ana, dia, tur in INDISP:
        a = ANALISTAS.index(ana); d = DIAS.index(dia); t = TURNOS.index(tur)
        m.addConstr(x[a, d, t] == 0)

    # (6) Senior só T3 ou T4
    for ana in SENIOR:
        a = ANALISTAS.index(ana)
        for d in range(D):
            m.addConstr(x[a, d, 0] == 0)
            m.addConstr(x[a, d, 1] == 0)

    return m, x

t0 = time.time()
mg, xg = montar_modelo_gurobi()
mg.optimize()
t_gurobi = time.time() - t0

# Comparar com CP-SAT (já rodado acima)
t_cpsat = solver.WallTime()

print(f'CP-SAT (OR-Tools): {t_cpsat*1000:7.1f} ms  status={solver.StatusName(status)}')
print(f'Gurobi MIP:        {t_gurobi*1000:7.1f} ms  status={mg.Status} (2=OPTIMAL)')
print()
print('Escala (Gurobi):')
print(f"{'Anal.':<10} | " + ' | '.join(f'{d:^6}' for d in DIAS))
for a in range(N):
    row = [f'{ANALISTAS[a]:<10}']
    for d in range(D):
        cell = ''
        for t in range(T):
            if xg[a,d,t].X > 0.5:
                cell = TURNOS[t]
        row.append(f'{cell:^6}')
    print(' | '.join(row))

### Comentário

Solução em <50 ms. Mas observe: **sem objetivo**, o solver empilha turnos em poucos analistas. Eduarda (senior) pode acabar com 0 turnos se ninguém forçar carga mínima.

Vamos atacar isso na extensão.

---

## Exercício de extensão: 7 dias + fairness + preferências

Ampliação:
- Adicionar sábado e domingo com **2 turnos** cada (T2 12–16h, T3 16–20h) — total 24 turnos/sem
- **Fairness:** minimizar $\max_a \text{carga}_a - \min_a \text{carga}_a$
- **Preferências:** cada analista lista 2 turnos preferidos; maximizar atendidas

Combinamos os dois objetivos em uma soma ponderada lexicográfica:
$$\min\ 100 \cdot \text{gap} - \text{prefs atendidas}$$

(O peso 100 garante que fairness vem primeiro; prefs entra como desempate.)

In [ ]:
DIAS_EXT = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
# Sáb e Dom: só T2 e T3 (12-16h e 16-20h)
TURNOS_DIA = {d: list(range(4)) if d < 5 else [1, 2] for d in range(7)}

PREFS = {
    'Ana':     [('Seg','T2'), ('Qua','T2')],
    'Bruno':   [('Qua','T1'), ('Qui','T1')],
    'Carla':   [('Qua','T3'), ('Sex','T2')],
    'Diego':   [('Seg','T1'), ('Ter','T1')],
    'Eduarda': [('Qui','T4'), ('Sex','T3')],
    'Fábio':   [('Sáb','T2'), ('Dom','T2')],
    'Gabi':    [('Sáb','T3'), ('Dom','T3')],
    'Hugo':    [('Seg','T4'), ('Sex','T4')],
}

D7 = len(DIAS_EXT)

m2 = cp_model.CpModel()
y = {}
for a in range(N):
    for d in range(D7):
        for t in TURNOS_DIA[d]:
            y[a,d,t] = m2.NewBoolVar(f'y[{a},{d},{t}]')

# Cobertura
for d in range(D7):
    for t in TURNOS_DIA[d]:
        m2.AddExactlyOne([y[a,d,t] for a in range(N)])

# Máximo 1 turno por dia
for a in range(N):
    for d in range(D7):
        if len(TURNOS_DIA[d]) > 1:
            m2.AddAtMostOne([y[a,d,t] for t in TURNOS_DIA[d]])

# Sem T4 → T1 no dia seguinte
for a in range(N):
    for d in range(D7 - 1):
        if 3 in TURNOS_DIA[d] and 0 in TURNOS_DIA[d+1]:
            m2.Add(y[a,d,3] + y[a,d+1,0] <= 1)

# Indisponibilidades
for ana, dia, tur in INDISP:
    a = ANALISTAS.index(ana); d = DIAS_EXT.index(dia); t = TURNOS.index(tur)
    if t in TURNOS_DIA[d]:
        m2.Add(y[a,d,t] == 0)

# Senior só T3/T4
for ana in SENIOR:
    a = ANALISTAS.index(ana)
    for d in range(D7):
        for t in TURNOS_DIA[d]:
            if t not in (2, 3):
                m2.Add(y[a,d,t] == 0)

# Carga por analista
carga = [sum(y[a,d,t] for d in range(D7) for t in TURNOS_DIA[d]) for a in range(N)]
c_max = m2.NewIntVar(0, 24, 'cmax')
c_min = m2.NewIntVar(0, 24, 'cmin')
for c in carga:
    m2.Add(c <= c_max)
    m2.Add(c >= c_min)

# Preferências
pref_vars = []
for ana, lista in PREFS.items():
    a = ANALISTAS.index(ana)
    for dia, tur in lista:
        d = DIAS_EXT.index(dia); t = TURNOS.index(tur)
        if t in TURNOS_DIA[d] and (a,d,t) in y:
            pref_vars.append(y[a,d,t])

# Objetivo lexicográfico: fairness primeiro, prefs depois
m2.Minimize((c_max - c_min) * 100 - sum(pref_vars))

solver = cp_model.CpSolver()
solver.parameters.num_search_workers = 4
status = solver.Solve(m2)
print(f'Status: {solver.StatusName(status)}  ({solver.WallTime()*1000:.1f} ms)')
print(f'Gap carga: {solver.Value(c_max) - solver.Value(c_min)}')
print(f'Prefs atendidas: {sum(solver.Value(p) for p in pref_vars)} / {len(pref_vars)}')
print()

# Imprime escala
print(f"{'Anal.':<8} | " + ' | '.join(f'{d:^6}' for d in DIAS_EXT))
for a in range(N):
    row = [f'{ANALISTAS[a]:<8}']
    for d in range(D7):
        cell = ''
        for t in TURNOS_DIA[d]:
            if (a,d,t) in y and solver.Value(y[a,d,t]) == 1:
                cell = TURNOS[t]
        row.append(f'{cell:^6}')
    print(' | '.join(row))

print('\nCargas finais:')
for a in range(N):
    print(f'  {ANALISTAS[a]:<10}: {sum(solver.Value(y[a,d,t]) for d in range(D7) for t in TURNOS_DIA[d] if (a,d,t) in y)} turnos')

### Para discutir

1. **Quanto tempo CP-SAT levou?** Compare com sua intuição de quanto MIP levaria.
2. **A solução é "justa"?** Cargas equilibradas (gap=0) e prefs atendidas (16/16) — é coincidência ou estrutura do problema?
3. **TODO — Comparação com MILP:** reescreva o mesmo problema no `pywraplp` (LinearSolver) e compare:
   - Número de variáveis e restrições
   - Tempo de solução
   - Legibilidade do código

4. **TODO — Stress test:** dobrar o número de analistas (16) ou adicionar 4 semanas (28 dias). CP-SAT ainda resolve em <1s?

In [ ]:
# TODO: implementar o mesmo modelo em MILP (pywraplp) e comparar
# Dica: AddAtMostOne(vars) em MILP = sum(vars) <= 1
# AddExactlyOne(vars) em MILP = sum(vars) == 1
# OnlyEnforceIf em MILP = big-M

---

## Quando CP-SAT, quando Gurobi MIP? — achado contra-intuitivo

Olhe os tempos acima — em **nosso problema pequeno** (8 analistas × 20 turnos = 160 binárias, restrições todas lineares), **Gurobi venceu CP-SAT em ordem de magnitude**. Por quê?

- Gurobi tem **presolve fortíssimo** (remove redundâncias antes de procurar)
- Relaxação LP é boa nesse problema (não tem big-M)
- Branch-and-cut + cortes automáticos resolvem em poucos nós
- CP-SAT carrega overhead de propagação que só compensa em escala

### A regrinha de bolso (calibrada):

| Situação | Ferramenta recomendada |
|---|---|
| **Problemas pequenos/médios com restrições lineares**, mesmo combinatórios | **Gurobi MIP** |
| Scheduling com **NoOverlap, AllDifferent, Cumulative** em escala | **CP-SAT** |
| **Variáveis intervalares** (tempo de início/fim/duração) | **CP-SAT** |
| Lot sizing, blending, network com fluxo contínuo | **Gurobi MIP** |
| Restrições verdadeiramente lógicas ("se X então Y, exceto se Z") | **CP-SAT** |
| Cliente já tem licença Gurobi e o problema é solúvel em MIP razoável | **Gurobi** |

### Conclusão honesta para a Gradus

Para muito do que aparece em consultoria brasileira de pequeno e médio porte, **Gurobi é melhor mesmo em problemas combinatórios** — só não é se o problema explode para milhares de tarefas com NoOverlap, Cumulative, etc.

**Discurso a usar com o cliente:** "a Genoa traz Gurobi porque cobre 90% dos casos. Para o resto (scheduling industrial massivo), existem ferramentas especializadas — e é nossa obrigação como consultoria recomendar a certa."